In [11]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, AutoConfig,
                          ElectraForSequenceClassification, ElectraConfig)
from nltk.sentiment import SentimentIntensityAnalyzer
from datasets import Dataset

In [12]:
data = pd.read_csv("cyberbullying.csv")
data = data[['tweet_text', 'cyberbullying_type']].dropna()

label_encoder = LabelEncoder()
data['label'] = label_encoder.fit_transform(data['cyberbullying_type'])

label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print("Label mapping:", label_mapping)

Label mapping: {'age': 0, 'ethnicity': 1, 'gender': 2, 'not_cyberbullying': 3, 'other_cyberbullying': 4, 'religion': 5}


In [13]:
class ModelOne:
    def __init__(self, model_name='cardiffnlp/twitter-roberta-base-sentiment'):
        config = AutoConfig.from_pretrained(model_name, num_labels=len(label_mapping))
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name, config=config, ignore_mismatched_sizes=True)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

    def fit(self, X, y=None): return self

    def transform(self, X, batch_size=32):
        self.model.eval()
        all_logits = []
        for i in range(0, len(X), batch_size):
            batch = X[i:i+batch_size]
            with torch.no_grad():
                inputs = self.tokenizer(batch.tolist(), return_tensors='pt', padding=True, truncation=True, max_length=512)
                outputs = self.model(**inputs)
                all_logits.append(outputs.logits.detach().cpu().numpy())
        return np.vstack(all_logits)

class ModelTwo:
    def __init__(self, model_name='google/electra-base-discriminator'):
        config = ElectraConfig.from_pretrained(model_name, num_labels=len(label_mapping))
        self.model = ElectraForSequenceClassification.from_pretrained(model_name, config=config, ignore_mismatched_sizes=True)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

    def fit(self, X, y=None): return self

    def transform(self, X, batch_size=32):
        self.model.eval()
        all_logits = []
        for i in range(0, len(X), batch_size):
            batch = X[i:i+batch_size]
            with torch.no_grad():
                inputs = self.tokenizer(batch.tolist(), return_tensors='pt', padding=True, truncation=True, max_length=512)
                outputs = self.model(**inputs)
                all_logits.append(outputs.logits.detach().cpu().numpy())
        return np.vstack(all_logits)

class ModelThree:
    def __init__(self):
        self.analyzer = SentimentIntensityAnalyzer()

    def fit(self, X, y=None): return self

    def transform(self, X):
        return np.array([list(self.analyzer.polarity_scores(x).values()) for x in X])

In [14]:
X_raw = data['tweet_text'].astype(str)
y = data['label']
X_train_raw, X_val_raw, y_train, y_val = train_test_split(X_raw, y, stratify=y, random_state=42)

In [15]:
roberta = ModelOne().fit(X_train_raw)
electra = ModelTwo().fit(X_train_raw)
vader = ModelThree().fit(X_train_raw)

roberta_train = roberta.transform(X_train_raw)
electra_train = electra.transform(X_train_raw)
vader_train = vader.transform(X_train_raw)

roberta_val = roberta.transform(X_val_raw)
electra_val = electra.transform(X_val_raw)
vader_val = vader.transform(X_val_raw)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment and are newly initialized because the shapes did not match:
- classifier.out_proj.weight: found shape torch.Size([3, 768]) in the checkpoint and torch.Size([6, 768]) in the model instantiated
- classifier.out_proj.bias: found shape torch.Size([3]) in the checkpoint and torch.Size([6]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at google/electra-base-discriminator and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [16]:
X_train_stacked = np.concatenate([roberta_train, electra_train, vader_train], axis=1)
X_val_stacked = np.concatenate([roberta_val, electra_val, vader_val], axis=1)

meta_model = LogisticRegression(max_iter=1000)
meta_model.fit(X_train_stacked, y_train)

LogisticRegression(max_iter=1000)

In [17]:
y_pred = meta_model.predict(X_val_stacked)
accuracy = np.mean(y_pred == y_val)
print(f"Stacking Accuracy: {accuracy:.4f}")

Stacking Accuracy: 0.4855


In [18]:
from sklearn.metrics import classification_report

# Generate classification report
print(classification_report(y_val, y_pred, target_names=label_encoder.classes_))

                     precision    recall  f1-score   support

                age       0.54      0.61      0.57      1998
          ethnicity       0.57      0.54      0.56      1990
             gender       0.43      0.41      0.42      1993
  not_cyberbullying       0.42      0.43      0.42      1986
other_cyberbullying       0.35      0.27      0.31      1956
           religion       0.56      0.65      0.60      2000

           accuracy                           0.49     11923
          macro avg       0.48      0.48      0.48     11923
       weighted avg       0.48      0.49      0.48     11923

